# 03 - Time Series Revenue Forecasting
In this notebook, we compare 6 time-series forecasting models (ARIMA, SARIMA, Prophet, XGBoost, LightGBM, and PyTorch LSTM) on out-of-sample test split and project future revenue.


In [ ]:
import pandas as pd
import pickle
import sys
sys.path.append('../src')

from preprocessing import load_merged_data
from feature_engineering import engineer_all_features
from forecasting import run_forecasting_pipeline

df = load_merged_data("../data")
df_feat = engineer_all_features(df)

# Run forecasting pipeline to train models and dump forecast_results.pkl
res = run_forecasting_pipeline(df_feat, "forecast_results.pkl")
print("Forecasting run complete.")


## Model Performance Comparison (Test Split)


In [ ]:
metrics_df = pd.DataFrame(res['metrics']).T
metrics_df = metrics_df[['MAE', 'RMSE', 'MAPE', 'R2']]
metrics_df.style.highlight_min(subset=['MAE', 'RMSE', 'MAPE'], color='#C8E6C9').highlight_max(subset=['R2'], color='#C8E6C9')


## Future Forecast (365 Days)


In [ ]:
forecast_df = pd.DataFrame(res['forecasts'])
print(f"Forecast dates shape: {forecast_df.shape}")
forecast_df.head()


## Plot Forecast Comparison


In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(14, 7))
plt.plot(pd.to_datetime(df_feat['Date'].tail(90)), df_feat['Revenue'].tail(90), label='Historical Revenue (Last 90d)', color='black', linewidth=2)
for model in ['Prophet', 'XGBoost', 'LightGBM', 'LSTM']:
    plt.plot(pd.to_datetime(forecast_df['Date']), forecast_df[model], label=f'{model} Forecast', alpha=0.8)
plt.title('Out-of-Sample Daily Revenue Forecast (Next 365 Days)', fontsize=14)
plt.ylabel('Revenue ($)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()
